# Lesson 25 Lab — Dynamic Shapes and Specialization

**Puzzle:** When runtime arguments, constexpr meta-parameters, tails, and cache keys change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates runtime arguments, constexpr meta-parameters, tails, and cache keys and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

A size used only in masks and grid calculation can remain a runtime argument, allowing one compiled kernel to cover many lengths. Values that shape tl.arange or control compile-time branches must be constexpr and create specialization boundaries.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["runtime arguments, constexpr meta-parameters, tails, and cache keys"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: named PyTorch CUDA/library or standard-grid path. Candidate: reviewed Triton kernel or explicit model described below.

Marking every shape value constexpr can create compilation churn; making every choice dynamic can prevent useful optimization.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 25
LESSON_TITLE = 'Dynamic Shapes and Specialization'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260838
}


## 5. Freeze the experiment

**Experiment:** Run one affine source over three lengths with different tails and retain first/warm results for each.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 3,
  "secondary": 0.019279999658465385,
  "max_abs_error": 4.76837158203125e-07,
  "passed": true,
  "details": {
    "shape_results": {
      "1025": {
        "first_ms": 331.23118290677667,
        "warm_median_ms": 0.019279999658465385,
        "tail": 1,
        "samples_ms": [
          0.027775999158620834,
          0.021824000403285027,
          0.01929599978029728,
          0.017632000148296356,
          0.019711999222636223,
          0.01961600035429001,
          0.01926399953663349,
          0.018271999433636665,
          0.01679999940097332,
          0.01708799973130226
        ]
      },
      "65537": {
        "first_ms": 0.09149312973022461,
        "warm_median_ms": 0.018383999355137348,
        "tail": 1,
        "samples_ms": [
          0.025087999179959297,
          0.020927999168634415,
          0.019231999292969704,
          0.020800000056624413,
          0.02163200080394745,
          0.017535999417304993,
          0.0162879992276430

## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Shapes passed | 3 |
| Largest warm median | 0.0193 ms |
| Maximum absolute error | 4.768e-07 |
| Acceptance gate | true |


## 8. Explain without overclaiming

One runtime-size kernel handled 3 shapes and three different tails without changing source; the largest warm median was 0.0193 ms.

A named Triton or PyTorch CUDA path executed on the recorded GPU. The result applies to the printed shape, dtype, implementation, and software stack; internal hardware causes require profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'native-backend',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Specialize only on values that change generated structure or produce a measured win large enough to repay compile cost.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 25,
  "title": "Dynamic Shapes and Specialization",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260838
  },
  "evidence_label": "native-backend",
  "metrics": {
    "primary": 3,
    "secondary": 0.019279999658465385,
    "max_abs_error": 4.76837158203125e-07,
    "passed": true,
    "details": {
      "shape_results": {
        "1025": {
          "first_ms": 331.23118290677667,
          "warm_median_ms": 0.019279999658465385,
          "tail": 1,
          "samples_ms": [
            0.027775999158620834,
            0.021824000403285027,
            0.01929599978029728,
            0.017632000148296356,
            0.019711999222636223,
            0.01961600035429001,
            0.01926399953663349,
            0.018271999433636665

## 10. Make the bounded decision

> Specialize only on values that change generated structure or produce a measured win large enough to repay compile cost.

**Failure analysis:** Marking every shape value constexpr can create compilation churn; making every choice dynamic can prevent useful optimization.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
